In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes transformers datasets

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
import json
from trl import SFTTrainer
from transformers import TrainingArguments

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Test dataset (replace with your people_data.json later)
data = [{"prompt": "While strolling through a botanical garden, Igor, now 20 earns a living as a tour guide.", "response": {"name": "Igor", "age": "20", "job": "tour guide", "gender": "male"}}]
ds = Dataset.from_list(data)

def format_ex(ex):
    resp = json.dumps(ex["response"])
    return {"text": tokenizer.apply_chat_template([{"role": "user", "content": ex["prompt"]}, {"role": "assistant", "content": resp}], tokenize=False)}

dataset = ds.map(format_ex)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer.train()

model.save_pretrained("lora_adapter")
model.save_pretrained_merged("phi3_finetuned_merged", tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf("phi3_finetuned_gguf", tokenizer, quantization_method="q4_k_m")

print("DONE! GGUF in phi3_finetuned_gguf/")